In [ ]:
import os, textwrap
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE = "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches"
TAG   = "2026-04-02"
DRIVE_BASE = f"{DRIVE}/{TAG}"

In [ ]:
# Audit existing zip directories on Drive — useful before re-packing.
!python -m training.audit_drive --drive-root {DRIVE} --run-tag {TAG}

In [ ]:
# Pack CNMF GT — full + patched (GT 192, LR 32) for training
!python -m spectral.pack_npy \
    --drive-root {DRIVE} \
    --run-tag {TAG} \
    --gt-source cnmf \
    --r2-thresh 0.5 \
    --out-full {DRIVE_BASE}/zips_cnmf_npy \
    --patched {DRIVE_BASE}/zips_cnmf_patched:192

In [ ]:
# Pack SFIM GT — same shape as CNMF, different fusion
!python -m spectral.pack_npy \
    --drive-root {DRIVE} \
    --run-tag {TAG} \
    --gt-source sfim \
    --r2-thresh 0.5 \
    --out-full {DRIVE_BASE}/zips_sfim_npy \
    --patched {DRIVE_BASE}/zips_sfim_patched:192

In [ ]:
# Synthetic baseline (EMIT source): real EMIT (96×96) as GT, bicubic-downsampled to 16×16 as LR.
# Used in the synthetic vs. real-pipeline ablation.
!python -m training.make_synthetic \
    --drive-root {DRIVE} \
    --run-tag {TAG} \
    --source emit \
    --method bicubic \
    --dst {DRIVE_BASE}/zips_synthetic_bicubic

In [ ]:
# Synthetic CNMF (full + patched 192): CNMF fusion (576×576) as GT,
# Gaussian-degraded to 96×96 as LR. Single-sensor synthetic comparison.
!python -m training.make_synthetic \
    --drive-root {DRIVE} \
    --run-tag {TAG} \
    --source cnmf \
    --r2-thresh 0.5 \
    --dst {DRIVE_BASE}/zips_synthetic_cnmf \
    --patched {DRIVE_BASE}/zips_synthetic_cnmf_patched:192